# Time Series Forecasting: ARIMA, SARIMA y Prophet

Descomposición de series temporales, tests de estacionariedad, y comparación de modelos
de forecasting sobre airline passengers (clásico) y ventas diarias sintéticas.

**Datasets:** Airline Passengers (Box-Jenkins) + Ventas diarias sintéticas  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import mean_squared_error, mean_absolute_error

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'   # naranja
ACCENT2 = '#6a9ad4'   # azul
ACCENT3 = '#2dc653'   # verde

print('Entorno listo.')

---
## 1. Datos

| Dataset | Fuente | Registros |
|---|---|---|
| Airline Passengers | statsmodels (clásico Box-Jenkins, 1949-1960) | 144 meses |
| Ventas diarias sintéticas | Generado con trend + estacionalidad + ruido | ~1095 días |

> **Nota:** Si no tienes los datos, el notebook genera datasets sintéticos automáticamente.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

# --- Airline Passengers ---
AIRLINE_PATH = Path('../data/processed/airline_passengers.csv')

if AIRLINE_PATH.exists():
    airline = pd.read_csv(AIRLINE_PATH, parse_dates=['date'])
    es_sintetico_airline = False
    print(f'[OK] Airline passengers cargado: {len(airline)} meses')
else:
    print('[INFO] Datos airline no encontrados. Generando sintéticos...')
    print('[TIP]  Ejecuta: python src/fetch_ts_data.py')
    from fetch_ts_data import generate_synthetic_airline
    airline = generate_synthetic_airline()
    es_sintetico_airline = True

airline = airline.set_index('date')
airline.index.freq = 'MS'

# --- Ventas diarias ---
SALES_PATH = Path('../data/processed/daily_sales.csv')

if SALES_PATH.exists():
    sales = pd.read_csv(SALES_PATH, parse_dates=['date'])
    print(f'[OK] Ventas diarias cargado: {len(sales)} días')
else:
    print('[INFO] Datos ventas no encontrados. Generando sintéticos...')
    from fetch_ts_data import generate_synthetic_daily_sales
    sales = generate_synthetic_daily_sales()

sales = sales.set_index('date')
sales.index.freq = 'D'

if es_sintetico_airline:
    print('\nDatos sintéticos. Patrones realistas pero no son datos oficiales.')

print(f'\nAirline: {airline.shape} | Sales: {sales.shape}')

# Vista rápida
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(airline.index, airline['passengers'], color=ACCENT, lw=1.5)
axes[0].set(title='Airline Passengers (mensual, 1949-1960)', ylabel='Pasajeros')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sales.index, sales['sales'], color=ACCENT2, lw=0.8, alpha=0.8)
axes[1].set(title='Ventas Diarias Sintéticas (2022-2024)', ylabel='Ventas ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Anatomía de una serie temporal

> **Pregunta:** ¿Qué componentes tiene una serie temporal?

Una serie temporal se puede descomponer en:
- **Trend:** tendencia a largo plazo
- **Seasonality:** patrón que se repite en ciclos regulares
- **Residual:** lo que queda después de eliminar trend + seasonality

Dos modelos de descomposición:
- **Aditivo:** y(t) = Trend + Seasonal + Residual (amplitud constante)
- **Multiplicativo:** y(t) = Trend x Seasonal x Residual (amplitud crece con el nivel)

In [ ]:
# 2.1 Descomposición de Airline Passengers
# Multiplicativa: la amplitud de la estacionalidad crece con el nivel
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

decomp_mult = seasonal_decompose(airline['passengers'], model='multiplicative', period=12)

for ax, data, title, color in [
    (axes[0], airline['passengers'], 'Observed', ACCENT),
    (axes[1], decomp_mult.trend, 'Trend', ACCENT2),
    (axes[2], decomp_mult.seasonal, 'Seasonal', ACCENT3),
    (axes[3], decomp_mult.resid, 'Residual', '#999'),
]:
    ax.plot(data, color=color, lw=1.5)
    ax.set(ylabel=title, title=title if ax == axes[0] else '')
    ax.grid(True, alpha=0.3)

axes[0].set_title('Descomposición Multiplicativa — Airline Passengers')
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Descomposición de Ventas Diarias
# Aditiva: la amplitud de la estacionalidad es más o menos constante
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

# Resamplear a semanal para descomposición más limpia
sales_weekly = sales['sales'].resample('W').mean()
decomp_add = seasonal_decompose(sales_weekly, model='additive', period=52)

for ax, data, title, color in [
    (axes[0], sales_weekly, 'Observed', ACCENT),
    (axes[1], decomp_add.trend, 'Trend', ACCENT2),
    (axes[2], decomp_add.seasonal, 'Seasonal', ACCENT3),
    (axes[3], decomp_add.resid, 'Residual', '#999'),
]:
    ax.plot(data, color=color, lw=1.2)
    ax.set(ylabel=title)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Descomposición Aditiva — Ventas Diarias (resampleado semanal)')
plt.tight_layout()
plt.show()

print('Airline Passengers: descomposición multiplicativa (amplitud crece con trend)')
print('Ventas Diarias: descomposición aditiva (amplitud estable)')

---
## 3. Estacionariedad

> **Pregunta:** ¿La serie es estacionaria? ¿Necesitamos transformarla?

Una serie es **estacionaria** si su media, varianza y autocorrelación no cambian con el tiempo.
ARIMA necesita series estacionarias. Dos tests formales:

- **ADF (Augmented Dickey-Fuller):** H0 = hay raíz unitaria (no estacionaria). p < 0.05 = estacionaria.
- **KPSS:** H0 = la serie es estacionaria. p < 0.05 = NO estacionaria.

In [ ]:
# 3.1 Tests de estacionariedad sobre Airline Passengers
def run_stationarity_tests(series, name=''):
    """Ejecuta ADF y KPSS y muestra resultados."""
    print(f'--- Tests de Estacionariedad: {name} ---')

    # ADF
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(series.dropna())
    print(f'  ADF Statistic: {adf_stat:.4f}')
    print(f'  ADF p-value:   {adf_p:.6f}')
    for key, val in adf_crit.items():
        print(f'    Crítico {key}: {val:.4f}')
    adf_ok = adf_p < 0.05
    print(f'  -> {"ESTACIONARIA" if adf_ok else "NO estacionaria"} (ADF)\n')

    # KPSS
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series.dropna(), regression='c')
    print(f'  KPSS Statistic: {kpss_stat:.4f}')
    print(f'  KPSS p-value:   {kpss_p:.4f}')
    kpss_ok = kpss_p >= 0.05
    print(f'  -> {"ESTACIONARIA" if kpss_ok else "NO estacionaria"} (KPSS)\n')

    return adf_ok, kpss_ok

adf_ok, kpss_ok = run_stationarity_tests(airline['passengers'], 'Airline Passengers (original)')

In [ ]:
# 3.2 Transformaciones para lograr estacionariedad
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Original
axes[0, 0].plot(airline['passengers'], color=ACCENT, lw=1.5)
axes[0, 0].set_title('Original')

# Log transform
airline_log = np.log(airline['passengers'])
axes[0, 1].plot(airline_log, color=ACCENT2, lw=1.5)
axes[0, 1].set_title('Log Transform')

# 1st difference
airline_diff1 = airline['passengers'].diff().dropna()
axes[0, 2].plot(airline_diff1, color=ACCENT3, lw=1.5)
axes[0, 2].set_title('1st Difference')

# Log + 1st difference
airline_log_diff1 = airline_log.diff().dropna()
axes[1, 0].plot(airline_log_diff1, color=ACCENT, lw=1.5)
axes[1, 0].set_title('Log + 1st Difference')

# 2nd difference
airline_diff2 = airline['passengers'].diff().diff().dropna()
axes[1, 1].plot(airline_diff2, color=ACCENT2, lw=1.5)
axes[1, 1].set_title('2nd Difference')

# Log + 1st diff + seasonal diff (lag 12)
airline_log_diff1_seasonal = airline_log_diff1.diff(12).dropna()
axes[1, 2].plot(airline_log_diff1_seasonal, color=ACCENT3, lw=1.5)
axes[1, 2].set_title('Log + Diff(1) + Diff(12)')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Transformaciones para Estacionariedad', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tests sobre log + 1st difference
print('Después de log + 1st difference:')
run_stationarity_tests(airline_log_diff1, 'Airline (log + diff1)')

print('Después de log + diff(1) + diff(12):')
run_stationarity_tests(airline_log_diff1_seasonal, 'Airline (log + diff1 + seasonal diff12)')

In [ ]:
# 3.3 ACF y PACF — Identificar órdenes AR y MA
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF y PACF de la serie log-diferenciada
plot_acf(airline_log_diff1.dropna(), ax=axes[0, 0], lags=36, color=ACCENT)
axes[0, 0].set_title('ACF — Airline (log + diff1)')

plot_pacf(airline_log_diff1.dropna(), ax=axes[0, 1], lags=36, color=ACCENT2, method='ywm')
axes[0, 1].set_title('PACF — Airline (log + diff1)')

# ACF y PACF de la serie completamente diferenciada (regular + seasonal)
plot_acf(airline_log_diff1_seasonal.dropna(), ax=axes[1, 0], lags=36, color=ACCENT)
axes[1, 0].set_title('ACF — Airline (log + diff1 + diff12)')

plot_pacf(airline_log_diff1_seasonal.dropna(), ax=axes[1, 1], lags=36, color=ACCENT2, method='ywm')
axes[1, 1].set_title('PACF — Airline (log + diff1 + diff12)')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Lectura ACF/PACF:')
print('  - ACF corta en lag k -> MA(q=k)')
print('  - PACF corta en lag k -> AR(p=k)')
print('  - Picos en lags 12, 24... -> estacionalidad (S=12)')
print('  - Esto nos guía hacia los órdenes (p,d,q) del ARIMA')

---
## 4. ARIMA

> **Pregunta:** ¿ARIMA puede predecir los próximos 24 meses de pasajeros?

ARIMA(p, d, q):
- **p** = orden autoregresivo (AR) — cuántos lags pasados usar
- **d** = orden de diferenciación — cuántas veces diferenciar para estacionariedad
- **q** = orden de media móvil (MA) — cuántos errores pasados usar

In [ ]:
# 4.1 Train/test split (últimos 24 meses como test)
train_size = len(airline) - 24
train_airline = airline.iloc[:train_size]
test_airline = airline.iloc[train_size:]

print(f'Train: {len(train_airline)} meses ({train_airline.index[0].strftime("%Y-%m")} a {train_airline.index[-1].strftime("%Y-%m")})')
print(f'Test:  {len(test_airline)} meses ({test_airline.index[0].strftime("%Y-%m")} a {test_airline.index[-1].strftime("%Y-%m")})')

# Trabajar con log para estabilizar varianza
train_log = np.log(train_airline['passengers'])
test_log = np.log(test_airline['passengers'])

In [ ]:
# 4.2 Grid search manual para ARIMA(p,d,q) sobre log(passengers)
best_aic = np.inf
best_order = None
results_arima = []

for p in range(0, 4):
    for d in range(0, 3):
        for q in range(0, 4):
            try:
                model = ARIMA(train_log, order=(p, d, q))
                fit = model.fit()
                results_arima.append({'order': (p, d, q), 'aic': fit.aic, 'bic': fit.bic})
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = (p, d, q)
            except Exception:
                continue

results_arima_df = pd.DataFrame(results_arima).sort_values('aic').head(10)
print(f'Top 10 modelos ARIMA por AIC:\n')
print(results_arima_df.to_markdown(index=False))
print(f'\nMejor ARIMA: {best_order} (AIC = {best_aic:.2f})')

In [ ]:
# 4.3 Fit del mejor ARIMA + forecast
arima_model = ARIMA(train_log, order=best_order)
arima_fit = arima_model.fit()

print(arima_fit.summary())

# Forecast 24 meses
arima_forecast_log = arima_fit.forecast(steps=24)
arima_forecast = np.exp(arima_forecast_log)

# Confidence intervals
arima_pred = arima_fit.get_forecast(steps=24)
arima_ci_log = arima_pred.conf_int()
arima_ci = np.exp(arima_ci_log)

In [ ]:
# 4.4 Diagnóstico de residuos ARIMA
residuals_arima = arima_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuos en el tiempo
axes[0, 0].plot(residuals_arima, color=ACCENT, lw=1)
axes[0, 0].axhline(0, color=ACCENT3, ls='--', lw=1)
axes[0, 0].set_title('Residuos en el Tiempo')
axes[0, 0].grid(True, alpha=0.3)

# Histograma
axes[0, 1].hist(residuals_arima, bins=25, color=ACCENT, alpha=0.85, edgecolor='#333')
axes[0, 1].axvline(0, color=ACCENT3, ls='--', lw=1.5)
axes[0, 1].set_title('Distribución de Residuos')

# ACF de residuos
plot_acf(residuals_arima, ax=axes[1, 0], lags=24, color=ACCENT2)
axes[1, 0].set_title('ACF de Residuos')

# Q-Q plot
stats.probplot(residuals_arima, dist='norm', plot=axes[1, 1])
axes[1, 1].get_lines()[0].set(color=ACCENT, markersize=3, alpha=0.5)
axes[1, 1].get_lines()[1].set(color=ACCENT2, lw=2)
axes[1, 1].set_title('Q-Q Plot')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Diagnóstico de Residuos — ARIMA{best_order}', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Ljung-Box test
lb_test = acorr_ljungbox(residuals_arima, lags=[10, 20], return_df=True)
print('Ljung-Box Test (H0: no hay autocorrelación en residuos):')
print(lb_test)
print(f'\nNormalidad (Shapiro-Wilk): p = {stats.shapiro(residuals_arima)[1]:.4f}')

In [ ]:
# 4.5 Predicted vs Actual — ARIMA
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train_airline.index, train_airline['passengers'], color='#666', lw=1, label='Train')
ax.plot(test_airline.index, test_airline['passengers'], color=ACCENT, lw=2, label='Real (test)')
ax.plot(test_airline.index, arima_forecast.values, color=ACCENT2, lw=2, ls='--', label=f'ARIMA{best_order}')
ax.fill_between(
    test_airline.index,
    arima_ci.iloc[:, 0],
    arima_ci.iloc[:, 1],
    alpha=0.15, color=ACCENT2, label='IC 95%'
)
ax.set(xlabel='Fecha', ylabel='Pasajeros', title=f'Forecast ARIMA{best_order} — Airline Passengers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas ARIMA
rmse_arima = np.sqrt(mean_squared_error(test_airline['passengers'], arima_forecast))
mae_arima = mean_absolute_error(test_airline['passengers'], arima_forecast)
mape_arima = np.mean(np.abs((test_airline['passengers'].values - arima_forecast.values) / test_airline['passengers'].values)) * 100

print(f'--- Métricas ARIMA{best_order} ---')
print(f'  RMSE: {rmse_arima:.2f}')
print(f'  MAE:  {mae_arima:.2f}')
print(f'  MAPE: {mape_arima:.2f}%')
print(f'  AIC:  {arima_fit.aic:.2f}')

---
## 5. SARIMA

> **Pregunta:** ¿Añadir estacionalidad al modelo mejora las predicciones?

SARIMA(p,d,q)(P,D,Q,s) extiende ARIMA con componentes estacionales:
- **(P,D,Q)** = órdenes AR, diferenciación y MA estacionales
- **s** = período estacional (12 para datos mensuales)

In [ ]:
# 5.1 Grid search SARIMA — búsqueda sobre parámetros estacionales
# Fijamos d=1, D=1, s=12 y buscamos p, q, P, Q
best_aic_sarima = np.inf
best_sarima_order = None
best_sarima_seasonal = None
results_sarima = []

for p in range(0, 3):
    for q in range(0, 3):
        for P in range(0, 3):
            for Q in range(0, 3):
                try:
                    model = SARIMAX(
                        train_log,
                        order=(p, 1, q),
                        seasonal_order=(P, 1, Q, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    fit = model.fit(disp=False, maxiter=200)
                    results_sarima.append({
                        'order': (p, 1, q),
                        'seasonal': (P, 1, Q, 12),
                        'aic': fit.aic,
                    })
                    if fit.aic < best_aic_sarima:
                        best_aic_sarima = fit.aic
                        best_sarima_order = (p, 1, q)
                        best_sarima_seasonal = (P, 1, Q, 12)
                except Exception:
                    continue

results_sarima_df = pd.DataFrame(results_sarima).sort_values('aic').head(10)
print(f'Top 10 modelos SARIMA por AIC:\n')
print(results_sarima_df.to_markdown(index=False))
print(f'\nMejor SARIMA: {best_sarima_order} x {best_sarima_seasonal} (AIC = {best_aic_sarima:.2f})')

In [ ]:
# 5.2 Fit del mejor SARIMA + forecast
sarima_model = SARIMAX(
    train_log,
    order=best_sarima_order,
    seasonal_order=best_sarima_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())

# Forecast
sarima_pred = sarima_fit.get_forecast(steps=24)
sarima_forecast_log = sarima_pred.predicted_mean
sarima_ci_log = sarima_pred.conf_int()
sarima_forecast = np.exp(sarima_forecast_log)
sarima_ci = np.exp(sarima_ci_log)

In [ ]:
# 5.2b Diagnóstico de residuos SARIMA
residuals_sarima = sarima_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuos en el tiempo
axes[0, 0].plot(residuals_sarima, color=ACCENT, lw=1)
axes[0, 0].axhline(0, color=ACCENT3, ls='--', lw=1)
axes[0, 0].set_title('Residuos en el Tiempo')
axes[0, 0].grid(True, alpha=0.3)

# Histograma
axes[0, 1].hist(residuals_sarima, bins=25, color=ACCENT, alpha=0.85, edgecolor='#333')
axes[0, 1].axvline(0, color=ACCENT3, ls='--', lw=1.5)
axes[0, 1].set_title('Distribución de Residuos')

# ACF de residuos
plot_acf(residuals_sarima, ax=axes[1, 0], lags=24, color=ACCENT2)
axes[1, 0].set_title('ACF de Residuos')

# Q-Q plot
stats.probplot(residuals_sarima, dist='norm', plot=axes[1, 1])
axes[1, 1].get_lines()[0].set(color=ACCENT, markersize=3, alpha=0.5)
axes[1, 1].get_lines()[1].set(color=ACCENT2, lw=2)
axes[1, 1].set_title('Q-Q Plot')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Diagnóstico de Residuos — SARIMA{best_sarima_order}x{best_sarima_seasonal}',
             y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Ljung-Box test
lb_test_sarima = acorr_ljungbox(residuals_sarima, lags=[10, 20], return_df=True)
print('Ljung-Box Test (H0: no hay autocorrelación en residuos):')
print(lb_test_sarima)
print(f'\nNormalidad (Shapiro-Wilk): p = {stats.shapiro(residuals_sarima)[1]:.4f}')

In [ ]:
# 5.3 Comparación ARIMA vs SARIMA
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train_airline.index, train_airline['passengers'], color='#666', lw=1, label='Train')
ax.plot(test_airline.index, test_airline['passengers'], color=ACCENT, lw=2, label='Real (test)')
ax.plot(test_airline.index, arima_forecast.values, color=ACCENT2, lw=2, ls='--', label=f'ARIMA{best_order}')
ax.plot(test_airline.index, sarima_forecast.values, color=ACCENT3, lw=2, ls='--', label=f'SARIMA{best_sarima_order}x{best_sarima_seasonal}')
ax.fill_between(
    test_airline.index,
    sarima_ci.iloc[:, 0],
    sarima_ci.iloc[:, 1],
    alpha=0.1, color=ACCENT3,
)
ax.set(xlabel='Fecha', ylabel='Pasajeros', title='ARIMA vs SARIMA — Airline Passengers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas SARIMA
rmse_sarima = np.sqrt(mean_squared_error(test_airline['passengers'], sarima_forecast))
mae_sarima = mean_absolute_error(test_airline['passengers'], sarima_forecast)
mape_sarima = np.mean(np.abs((test_airline['passengers'].values - sarima_forecast.values) / test_airline['passengers'].values)) * 100

print(f'--- Comparación ARIMA vs SARIMA ---')
print(f'{"Métrica":>8s}  {"ARIMA":>10s}  {"SARIMA":>10s}  {"Mejor":>8s}')
print(f'{"RMSE":>8s}  {rmse_arima:>10.2f}  {rmse_sarima:>10.2f}  {"SARIMA" if rmse_sarima < rmse_arima else "ARIMA":>8s}')
print(f'{"MAE":>8s}  {mae_arima:>10.2f}  {mae_sarima:>10.2f}  {"SARIMA" if mae_sarima < mae_arima else "ARIMA":>8s}')
print(f'{"MAPE":>8s}  {mape_arima:>9.2f}%  {mape_sarima:>9.2f}%  {"SARIMA" if mape_sarima < mape_arima else "ARIMA":>8s}')
print(f'{"AIC":>8s}  {arima_fit.aic:>10.2f}  {sarima_fit.aic:>10.2f}  {"SARIMA" if sarima_fit.aic < arima_fit.aic else "ARIMA":>8s}')

---
## 6. Prophet

> **Pregunta:** ¿Prophet de Meta captura mejor la estacionalidad que ARIMA?

Prophet funciona especialmente bien con:
- Datos diarios con múltiples estacionalidades (semanal + anual)
- Tendencias no lineales
- Datos con missing values y outliers

Lo probamos con el dataset de ventas diarias.

In [ ]:
# 6.1 Preparar datos para Prophet (requiere columnas 'ds' y 'y')
sales_prophet = sales[['sales']].reset_index()
sales_prophet.columns = ['ds', 'y']

# Train/test split (último 20%)
split_idx = int(len(sales_prophet) * 0.8)
train_prophet = sales_prophet.iloc[:split_idx].copy()
test_prophet = sales_prophet.iloc[split_idx:].copy()

print(f'Train: {len(train_prophet)} días ({train_prophet["ds"].iloc[0].strftime("%Y-%m-%d")} a {train_prophet["ds"].iloc[-1].strftime("%Y-%m-%d")})')
print(f'Test:  {len(test_prophet)} días ({test_prophet["ds"].iloc[0].strftime("%Y-%m-%d")} a {test_prophet["ds"].iloc[-1].strftime("%Y-%m-%d")})')

In [ ]:
# 6.2 Fit Prophet
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
prophet_model.fit(train_prophet)

# Forecast sobre todo el período (train + test)
future = prophet_model.make_future_dataframe(periods=len(test_prophet), freq='D')
prophet_forecast = prophet_model.predict(future)

print(f'Prophet fit completo. Forecast generado: {len(prophet_forecast)} días')

In [ ]:
# 6.3 Componentes de Prophet (trend + estacionalidades)
# Recreamos el plot manualmente para respetar el dark theme
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# Trend
axes[0].plot(prophet_forecast['ds'], prophet_forecast['trend'], color=ACCENT, lw=2)
axes[0].fill_between(
    prophet_forecast['ds'],
    prophet_forecast['trend_lower'],
    prophet_forecast['trend_upper'],
    color=ACCENT, alpha=0.15,
)
axes[0].set_title('Trend')
axes[0].grid(True, alpha=0.3)

# Weekly seasonality
weekly = prophet_forecast[['ds', 'weekly']].copy()
weekly['dow'] = weekly['ds'].dt.dayofweek
weekly_avg = weekly.groupby('dow')['weekly'].mean()
axes[1].plot(
    ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    weekly_avg.values,
    color=ACCENT2, lw=2, marker='o', markersize=5,
)
axes[1].set_title('Weekly Seasonality')
axes[1].grid(True, alpha=0.3)

# Yearly seasonality
yearly = prophet_forecast[['ds', 'yearly']].copy()
yearly['doy'] = yearly['ds'].dt.dayofyear
yearly_avg = yearly.groupby('doy')['yearly'].mean()
axes[2].plot(yearly_avg.index, yearly_avg.values, color=ACCENT3, lw=1.5)
axes[2].set_title('Yearly Seasonality')
axes[2].set_xlabel('Day of Year')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Componentes Prophet — Ventas Diarias', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Forecast plot — Prophet
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train_prophet['ds'], train_prophet['y'], color='#666', lw=0.5, alpha=0.6, label='Train')
ax.plot(test_prophet['ds'], test_prophet['y'], color=ACCENT, lw=1, label='Real (test)')

# Prophet forecast solo en período de test
prophet_test = prophet_forecast[prophet_forecast['ds'].isin(test_prophet['ds'])]
ax.plot(prophet_test['ds'], prophet_test['yhat'], color=ACCENT3, lw=1.5, ls='--', label='Prophet')
ax.fill_between(
    prophet_test['ds'],
    prophet_test['yhat_lower'],
    prophet_test['yhat_upper'],
    alpha=0.1, color=ACCENT3, label='IC 95%'
)

ax.set(xlabel='Fecha', ylabel='Ventas ($)', title='Prophet Forecast — Ventas Diarias')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas Prophet
prophet_test_merged = test_prophet.merge(prophet_test[['ds', 'yhat']], on='ds')
rmse_prophet = np.sqrt(mean_squared_error(prophet_test_merged['y'], prophet_test_merged['yhat']))
mae_prophet = mean_absolute_error(prophet_test_merged['y'], prophet_test_merged['yhat'])
mape_prophet = np.mean(np.abs((prophet_test_merged['y'].values - prophet_test_merged['yhat'].values) / prophet_test_merged['y'].values)) * 100

print(f'--- Métricas Prophet (ventas diarias) ---')
print(f'  RMSE: {rmse_prophet:.2f}')
print(f'  MAE:  {mae_prophet:.2f}')
print(f'  MAPE: {mape_prophet:.2f}%')

In [ ]:
# 6.5 Cross-validation con Prophet
print('Ejecutando cross-validation de Prophet...')
cv_results = cross_validation(
    prophet_model,
    initial='365 days',
    period='60 days',
    horizon='90 days',
)

cv_metrics = performance_metrics(cv_results)

# Display con MAPE en porcentaje
cv_display = cv_metrics[['horizon', 'rmse', 'mae', 'mape']].tail(10).copy()
cv_display['mape'] = cv_display['mape'] * 100
cv_display = cv_display.rename(columns={'mape': 'mape (%)'})
print(f'\n--- Cross-Validation Prophet ---')
print(f'Nota: MAPE expresado en porcentaje (%)')
print(cv_display.to_markdown(index=False))

# Plot MAPE por horizonte
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cv_metrics['horizon'].dt.days, cv_metrics['mape'] * 100, color=ACCENT, lw=2)
ax.set(xlabel='Horizonte (días)', ylabel='MAPE (%)', title='MAPE por Horizonte de Predicción — Prophet CV')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Comparación de modelos

> **Pregunta:** ¿Qué modelo funciona mejor y cuándo usar cada uno?

Para comparar los 3 modelos en igualdad de condiciones, usamos ARIMA y SARIMA también
sobre las ventas diarias (resampleadas a semanal para que los modelos converjan).

In [ ]:
# 7.1 ARIMA y SARIMA sobre ventas diarias (resampleado semanal)
sales_weekly_full = sales['sales'].resample('W').mean()
split_w = int(len(sales_weekly_full) * 0.8)
train_w = sales_weekly_full.iloc[:split_w]
test_w = sales_weekly_full.iloc[split_w:]

# ARIMA sobre ventas semanales
best_aic_w = np.inf
best_order_w = None
for p in range(0, 4):
    for d in range(0, 3):
        for q in range(0, 4):
            try:
                fit = ARIMA(train_w, order=(p, d, q)).fit()
                if fit.aic < best_aic_w:
                    best_aic_w = fit.aic
                    best_order_w = (p, d, q)
            except Exception:
                continue

arima_w_fit = ARIMA(train_w, order=best_order_w).fit()
arima_w_forecast = arima_w_fit.forecast(steps=len(test_w))

# SARIMA sobre ventas semanales (s=52 semanas)
best_aic_sw = np.inf
best_order_sw = None
best_seasonal_sw = None
for p in range(0, 3):
    for q in range(0, 3):
        for P in range(0, 2):
            for Q in range(0, 2):
                try:
                    fit = SARIMAX(
                        train_w, order=(p, 1, q),
                        seasonal_order=(P, 1, Q, 52),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    ).fit(disp=False, maxiter=200)
                    if fit.aic < best_aic_sw:
                        best_aic_sw = fit.aic
                        best_order_sw = (p, 1, q)
                        best_seasonal_sw = (P, 1, Q, 52)
                except Exception:
                    continue

sarima_w_fit = SARIMAX(
    train_w, order=best_order_sw,
    seasonal_order=best_seasonal_sw,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_w_forecast = sarima_w_fit.get_forecast(steps=len(test_w)).predicted_mean

# Prophet sobre ventas semanales (ya calculado arriba sobre diario, resampleamos predicciones)
prophet_weekly = prophet_forecast.set_index('ds')['yhat'].resample('W').mean()
prophet_w_test = prophet_weekly.reindex(test_w.index)

print(f'ARIMA semanal: {best_order_w}')
print(f'SARIMA semanal: {best_order_sw} x {best_seasonal_sw}')
print(f'Test: {len(test_w)} semanas')

In [ ]:
# 7.2 Tabla de métricas comparativa
def calc_metrics(actual, predicted, name):
    """Calcula RMSE, MAE, MAPE para un modelo."""
    mask = ~(actual.isna() | predicted.isna())
    a, p = actual[mask].values, predicted[mask].values
    rmse = np.sqrt(mean_squared_error(a, p))
    mae = mean_absolute_error(a, p)
    mape = np.mean(np.abs((a - p) / a)) * 100
    return {'Modelo': name, 'RMSE': f'{rmse:.2f}', 'MAE': f'{mae:.2f}', 'MAPE': f'{mape:.2f}%'}

metrics_table = pd.DataFrame([
    calc_metrics(test_w, arima_w_forecast, f'ARIMA{best_order_w}'),
    calc_metrics(test_w, sarima_w_forecast, f'SARIMA{best_order_sw}x{best_seasonal_sw}'),
    calc_metrics(test_w, prophet_w_test, 'Prophet'),
])

print('--- Comparación de Modelos (ventas semanales, test set) ---\n')
print(metrics_table.to_markdown(index=False))

In [ ]:
# 7.3 Visualización final: 3 modelos superpuestos
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_w.index, train_w.values, color='#555', lw=0.8, alpha=0.5, label='Train')
ax.plot(test_w.index, test_w.values, color=ACCENT, lw=2, label='Real (test)')
ax.plot(test_w.index, arima_w_forecast.values, color=ACCENT2, lw=1.5, ls='--', label=f'ARIMA{best_order_w}')
ax.plot(test_w.index, sarima_w_forecast.values, color=ACCENT3, lw=1.5, ls='--', label=f'SARIMA')
ax.plot(prophet_w_test.index, prophet_w_test.values, color='#ff6b9d', lw=1.5, ls='--', label='Prophet')

ax.set(xlabel='Fecha', ylabel='Ventas medias semanales ($)', title='Comparación de Modelos — Ventas Semanales')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Síntesis y conclusiones

| Modelo | Mejor para... | Limitaciones |
|---|---|---|
| **ARIMA** | Series sin estacionalidad clara, datos pequeños | No captura patrones estacionales |
| **SARIMA** | Series con estacionalidad regular y conocida | Lento en grid search, requiere elegir (P,D,Q,s) |
| **Prophet** | Datos diarios con múltiples estacionalidades, outliers, missing data | Caja negra, menos control sobre el modelo |

In [ ]:
# 8.1 Resumen final
print('=' * 60)
print('  RESUMEN — TIME SERIES FORECASTING')
print('=' * 60)

print('\n--- Airline Passengers (ARIMA vs SARIMA) ---')
print(f'  ARIMA{best_order}:')
print(f'    RMSE = {rmse_arima:.2f} | MAE = {mae_arima:.2f} | MAPE = {mape_arima:.2f}%')
print(f'  SARIMA{best_sarima_order}x{best_sarima_seasonal}:')
print(f'    RMSE = {rmse_sarima:.2f} | MAE = {mae_sarima:.2f} | MAPE = {mape_sarima:.2f}%')

print('\n--- Ventas Diarias (3 modelos) ---')
print(metrics_table.to_markdown(index=False))

print('\n--- Conclusiones ---')
print('1. ARIMA sin componente estacional no captura los picos/valles cíclicos.')
print('2. SARIMA mejora significativamente al modelar la estacionalidad explícitamente.')
print('3. Prophet funciona bien out-of-the-box para datos diarios con múltiples patrones.')
print('4. La elección del modelo depende del tipo de datos y el caso de uso.')
print('5. Siempre validar con métricas sobre un test set real, no solo AIC/BIC.')